In [2]:
import torch
import torch.nn as nn
import numpy as np
import pickle
import json
from sklearn.naive_bayes import GaussianNB
from IDS import IntrusionDetector
from simple_rules import AttackPlausibilityChecker

# --- 1. Load preprocessed data and model ---
X_train = np.load('results/X_train.npy')
X_test = np.load('results/X_test.npy')
y_train = np.load('results/y_train.npy')
y_test = np.load('results/y_test.npy')
train_original_labels = np.load('results/train_original_labels.npy', allow_pickle=True)
test_original_labels = np.load('results/test_original_labels.npy', allow_pickle=True)

with open('results/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('results/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)
with open('nslkdd_features.json', 'r') as f:
    features_meta = json.load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = IntrusionDetector(input_dim=X_train.shape[1]).to(device)
model.load_state_dict(torch.load('ids_model.pth', map_location=device))
model.eval()

# --- 2. Train GNB on original multi-class labels ---
gnb = GaussianNB()
gnb.fit(X_train, train_original_labels)

# Extract GNB parameters for differentiable log-likelihood
# gnb.theta_[class_idx, feature_idx] = mean
# gnb.var_[class_idx, feature_idx] = variance
gnb_classes = list(gnb.classes_)
gnb_means = torch.tensor(gnb.theta_, dtype=torch.float32, device=device)  # (n_classes, n_features)
gnb_vars = torch.tensor(gnb.var_, dtype=torch.float32, device=device)  # (n_classes, n_features)

print(f"GNB trained with {len(gnb_classes)} classes: {gnb_classes}")

# --- 3. Build modifiability masks ---
modifiability_map = {}
for feat in features_meta:
    modifiability_map[feat['name']] = feat['modifiability']

highly_modifiable_mask = np.zeros(len(feature_names), dtype=bool)
partially_modifiable_mask = np.zeros(len(feature_names), dtype=bool)

for i, fname in enumerate(feature_names):
    for orig_name in modifiability_map:
        if fname == orig_name or fname.startswith(orig_name + '_'):
            mod_level = modifiability_map[orig_name]
            if mod_level == 'HIGHLY_MODIFIABLE':
                highly_modifiable_mask[i] = True
            elif mod_level == 'PARTIALLY_MODIFIABLE':
                partially_modifiable_mask[i] = True
            break

combined_modifiable_mask = highly_modifiable_mask | partially_modifiable_mask

# --- 4. Feature ranges from training data ---
feature_min = X_train.min(axis=0)
feature_max = X_train.max(axis=0)

# --- 5. Identify true positives (excluding zero-day attacks) ---
# Find zero-day attack types
train_attack_types = set(train_original_labels[train_original_labels != 'normal'])
zero_day_types = set(test_original_labels[test_original_labels != 'normal']) - train_attack_types

X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
with torch.no_grad():
    preds = model(X_test_t).cpu().numpy().flatten()
    pred_labels = (preds >= 0.5).astype(int)

# True positives that are NOT zero-day
is_zero_day = np.isin(test_original_labels, list(zero_day_types))
true_positive_non_zd = (y_test == 1) & (pred_labels == 1) & (~is_zero_day)
tp_indices = np.where(true_positive_non_zd)[0]

print(f"Zero-day attack types: {sorted(zero_day_types)}")
print(f"True positives (non-zero-day): {len(tp_indices)}")

# Attack type mapping for plausibility
attack_type_map = {}
for feat in features_meta:
    if feat['name'] == 'label':
        for label, category in feat['flag_values'].items():
            if 'DoS' in category:
                attack_type_map[label] = 'DoS'
            elif 'Probe' in category:
                attack_type_map[label] = 'Probe'
            elif 'R2L' in category:
                attack_type_map[label] = 'R2L'
            elif 'U2R' in category:
                attack_type_map[label] = 'U2R'


# --- 6. Differentiable GNB log-likelihood ---
def gnb_log_likelihood(x, class_label):
    """
    Compute log p(x|y) for a given class using GNB parameters.
    x: (1, n_features) tensor
    class_label: string label of the attack class
    Returns: scalar tensor (log-likelihood)
    """
    class_idx = gnb_classes.index(class_label)
    mu = gnb_means[class_idx]  # (n_features,)
    var = gnb_vars[class_idx]  # (n_features,)

    # log p(x_i | y) = -0.5 * log(2*pi*var_i) - (x_i - mu_i)^2 / (2*var_i)
    log_probs = -0.5 * torch.log(2 * np.pi * var) - (x.squeeze(0) - mu) ** 2 / (2 * var)
    return log_probs.sum()


# --- 7. PGD with combined loss ---
def pgd_attack_with_plausibility(model, x_orig, modifiable_mask, epsilon,
                                 attack_label, alpha=0.01, num_iter=40,
                                 feat_min=None, feat_max=None, lam=0.1):
    """
    PGD with combined loss: L_total = L_classifier + lambda * L_plausibility
    L_classifier = model output (minimize to push toward normal)
    L_plausibility = -log p(x|y_attack) (minimize to stay on attack manifold)
    """
    x_adv = torch.tensor(x_orig.copy(), dtype=torch.float32, device=device).unsqueeze(0)
    x_orig_t = torch.tensor(x_orig.copy(), dtype=torch.float32, device=device).unsqueeze(0)

    mask_t = torch.tensor(modifiable_mask, dtype=torch.float32, device=device)
    feat_min_t = torch.tensor(feat_min, dtype=torch.float32, device=device)
    feat_max_t = torch.tensor(feat_max, dtype=torch.float32, device=device)

    for _ in range(num_iter):
        x_adv.requires_grad_(True)

        # L_classifier: model output (want to minimize → push to normal)
        output = model(x_adv)
        L_classifier = output.squeeze()

        # L_plausibility: negative log-likelihood under GNB for original attack class
        log_lik = gnb_log_likelihood(x_adv, attack_label)
        L_plausibility = -log_lik

        # Combined loss
        L_total = L_classifier + lam * L_plausibility
        L_total.backward()

        with torch.no_grad():
            grad = x_adv.grad.sign()
            perturbation = -alpha * grad * mask_t
            x_adv = x_adv + perturbation

            # Project onto epsilon ball
            delta = x_adv - x_orig_t
            delta = torch.clamp(delta, -epsilon, epsilon)
            delta = delta * mask_t
            x_adv = x_orig_t + delta

            # Clamp to valid feature ranges
            x_adv = torch.max(x_adv, feat_min_t.unsqueeze(0))
            x_adv = torch.min(x_adv, feat_max_t.unsqueeze(0))

            x_adv = x_adv.detach()

    return x_adv.squeeze(0).cpu().numpy()


# --- 8. Run attacks ---
epsilons = [0.05, 0.1, 0.15, 0.2, 0.3]
checker = AttackPlausibilityChecker(feature_names)

print(f"\n{'=' * 70}")
print(f"PGD with Plausibility Loss (lambda=0.1) Results")
print(f"{'=' * 70}")

for eps in epsilons:
    successful_attacks = 0
    plausible_attacks = 0
    total_attacks = len(tp_indices)

    for idx in tp_indices:
        x_orig = X_test[idx]
        orig_label = test_original_labels[idx]
        atk_type = attack_type_map.get(orig_label, 'DoS')

        # Try with highly modifiable first
        x_adv = pgd_attack_with_plausibility(
            model, x_orig, highly_modifiable_mask, epsilon=eps,
            attack_label=orig_label, alpha=0.01, num_iter=40,
            feat_min=feature_min, feat_max=feature_max, lam=0.1
        )

        x_adv_t = torch.tensor(x_adv, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            pred = model(x_adv_t).item()

        # If not successful, try combined mask
        if pred >= 0.5:
            x_adv = pgd_attack_with_plausibility(
                model, x_orig, combined_modifiable_mask, epsilon=eps,
                attack_label=orig_label, alpha=0.01, num_iter=40,
                feat_min=feature_min, feat_max=feature_max, lam=0.1
            )
            x_adv_t = torch.tensor(x_adv, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                pred = model(x_adv_t).item()

        if pred < 0.5:
            successful_attacks += 1

            is_plausible, violations = checker.check_plausibility(
                x_adv, attack_type=atk_type, scaler=scaler, verbose=False
            )
            if is_plausible:
                plausible_attacks += 1

    success_rate = successful_attacks / total_attacks * 100 if total_attacks > 0 else 0
    plausible_rate = plausible_attacks / successful_attacks * 100 if successful_attacks > 0 else 0

    print(f"\nEpsilon: {eps}")
    print(f"  Total attacks attempted:  {total_attacks}")
    print(f"  Successful attacks:       {successful_attacks} ({success_rate:.2f}%)")
    print(f"  Plausible attacks:        {plausible_attacks} ({plausible_rate:.2f}% of successful)")

GNB trained with 23 classes: [np.str_('back'), np.str_('buffer_overflow'), np.str_('ftp_write'), np.str_('guess_passwd'), np.str_('imap'), np.str_('ipsweep'), np.str_('land'), np.str_('loadmodule'), np.str_('multihop'), np.str_('neptune'), np.str_('nmap'), np.str_('normal'), np.str_('perl'), np.str_('phf'), np.str_('pod'), np.str_('portsweep'), np.str_('rootkit'), np.str_('satan'), np.str_('smurf'), np.str_('spy'), np.str_('teardrop'), np.str_('warezclient'), np.str_('warezmaster')]
Zero-day attack types: ['apache2', 'httptunnel', 'mailbomb', 'mscan', 'named', 'processtable', 'ps', 'saint', 'sendmail', 'snmpgetattack', 'snmpguess', 'sqlattack', 'udpstorm', 'worm', 'xlock', 'xsnoop', 'xterm']
True positives (non-zero-day): 6995

PGD with Plausibility Loss (lambda=0.1) Results

Epsilon: 0.05
  Total attacks attempted:  6995
  Successful attacks:       58 (0.83%)
  Plausible attacks:        19 (32.76% of successful)

Epsilon: 0.1
  Total attacks attempted:  6995
  Successful attacks:     